In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import json

from SharedModules import input_dir, output_dir, model_dir
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.base import clone

warnings.filterwarnings("ignore")

_train = pd.read_csv(input_dir + "train.csv", index_col=0)
_test = pd.read_csv(input_dir + "test.csv", index_col=0)

target = "Heart Disease"

cv = KFold(n_splits=10, shuffle=True, random_state=42)

X = _train.drop(target, axis=1)
y = _train[target].map({"Absence": 0, "Presence": 1})
X_test = _test

params_default = {
    "boosting_type": "gbdt",
    "device": "gpu",
    "verbose": -1,
}

with open(model_dir + "lgbm_base_params.json") as f:
    best_params = json.load(f)

params = params_default | best_params

lgbm = LGBMClassifier(**params)

In [ ]:
oof = np.zeros(len(X))
oof_scores = []

for fold, (idx_tr, idx_val) in enumerate(cv.split(X, y)):
    X_tr, X_val = X.iloc[idx_tr], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]

    model = clone(lgbm)
    model.fit(X_tr, y_tr)
    y_pred = model.predict_proba(X_val)[:, -1]
    score = roc_auc_score(y_val, y_pred)

    oof[idx_val] = y_pred
    oof_scores.append(score)

    print(f"fold {fold + 1} score: {score:.4f}")

print(f"Average Score: {np.mean(oof_scores): .4f}")

df_oof = pd.DataFrame(data=oof, index=X.index, columns=["LGBM_Base"])
df_oof.to_csv(model_dir + "LGBM/oof.csv")

fold 1 score: 0.9559
fold 2 score: 0.9550
fold 3 score: 0.9556
fold 4 score: 0.9558
fold 5 score: 0.9548
fold 6 score: 0.9577
fold 7 score: 0.9553
fold 8 score: 0.9550
fold 9 score: 0.9557
fold 10 score: 0.9551
Average Score:  0.9556


In [3]:
lgbm.fit(X, y)
sub = pd.DataFrame(data=lgbm.predict_proba(X_test)[:, -1], index=X_test.index, columns=["LGBM_Base"])
sub.to_csv(output_dir + "submission_lgbm.csv")